In [1]:
print("Hello world")

Hello world


In [2]:
import sys
print(sys.executable)

/home/faytimen/Documents/Projects/dara/.venv/bin/python


In [3]:
import dara
print(dara.__file__)

/home/faytimen/Documents/Projects/dara/src/dara/__init__.py


If running this on a new computer ect. run this first and check for any errors and if the file paths are in the right place. 

# DARA Fork + Environment Workflow

Reference notes for setting up and maintaining a personal fork of
[CederGroupHub/dara](https://github.com/CederGroupHub/dara), with an isolated
Python 3.13 environment, while staying able to pull in upstream updates
cleanly.

---

## 1. Fork & clone

1. Fork `CederGroupHub/dara` on GitHub → creates `github.com/<your-username>/dara`.
2. Clone **your fork** (not the original):
   ```bash
   git clone https://github.com/<your-username>/dara
   cd dara
   ```
3. Add the original repo as a second remote, `upstream`:
   ```bash
   git remote add upstream https://github.com/CederGroupHub/dara.git
   ```
4. Verify both remotes exist:
   ```bash
   git remote -v
   # origin      -> your fork (fetch/push)
   # upstream    -> CederGroupHub/dara (fetch/push)
   ```

---

## 2. Branching model

- **`main`** = a clean mirror of upstream. Never commit directly to it.
- **`dev`** (or any name you like) = where your actual work happens.

Create your working branch:
```bash
git checkout -b dev
```

Habit: run `git branch` before editing anything, to confirm you're on `dev`.

---

## 3. Python 3.13 via pyenv

Installed pyenv to manage Python versions without touching the system
Python or needing `sudo` after initial setup.

```bash
# one-time build dependencies
sudo apt update
sudo apt install -y make build-essential libssl-dev zlib1g-dev \
libbz2-dev libreadline-dev libsqlite3-dev wget curl llvm \
libncursesw5-dev xz-utils tk-dev libxml2-dev libxmlsec1-dev libffi-dev liblzma-dev

# install pyenv
curl https://pyenv.run | bash
```

Added to **both** `~/.profile` and `~/.bashrc`:
```bash
export PYENV_ROOT="$HOME/.pyenv"
[[ -d $PYENV_ROOT/bin ]] && export PATH="$PYENV_ROOT/bin:$PATH"
eval "$(pyenv init - bash)"
```
`~/.bashrc` additionally gets:
```bash
eval "$(pyenv virtualenv-init -)"
```

Reload shell, then install Python 3.13.15 (avoid the `t` / free-threaded
variants — less compatible with compiled scientific packages):
```bash
source ~/.bashrc
pyenv install 3.13.15
```

Pin it to this project only (does not touch global Python):
```bash
cd ~/Documents/Projects/dara
pyenv local 3.13.15
```
This writes a `.python-version` file. DARA's own `.gitignore` already
excludes this file, so it stays local to your machine only — fine, since
it's not collaborative yet.

---

## 4. Virtual environment + editable install

```bash
python -m venv .venv          # uses 3.13.15, thanks to pyenv local
source .venv/bin/activate
pip install -e .
```

Editable install means:
- `git pull` updates code instantly — no reinstall needed.
- Changing dependencies in `pyproject.toml` **does** require re-running
  `pip install -e .`.

Sanity check inside a notebook/script:
```python
import sys
print(sys.executable)   # should point into .venv

import dara
print(dara.__file__)    # should point into src/dara/, NOT site-packages
```
If `dara.__file__` points into `src/dara/`, the editable install is
correctly wired to your repo's source.

**Notebook gotcha:** Jupyter kernels can silently use a different Python
than your terminal's active venv. Always check `sys.executable` in a
notebook before assuming it matches your `.venv`.

---

## 5. Everyday update workflow

```bash
cd dara
git pull
pip install -e .
```
Both lines matter — `git pull` alone won't pick up new/changed
dependencies from `pyproject.toml`.

---

## 6. Pushing your own work

```bash
git branch                     # confirm you're on dev
git status                     # see what's changed
git add <folder-or-file>
git commit -m "clear message"
git push origin dev
```

---

## 7. Checking & pulling upstream changes

Check what's new *without* touching your files:
```bash
git fetch upstream
git log main..upstream/main --oneline
```
Empty output = you're already current.

If there are new commits, update your mirror branch and sync it to your
fork on GitHub:
```bash
git checkout main
git merge upstream/main        # always a clean fast-forward, since main is never edited directly
git push origin main
```

Then bring those changes into your working branch:
```bash
git checkout dev
git merge main
```
This is the only place conflicts can happen — and only if you've edited a
line that upstream also changed.

**Note:** DARA also cuts version tags (e.g. `v1.3.0`). If tracking `main`
ever feels too unstable, a tagged release is a pinned alternative:
```bash
git checkout v1.3.0
```

---

## Notes

- Optional dependency warnings (`ipywidgets` for tqdm/Ray notebook output)
  are cosmetic — fix with `pip install ipywidgets` + kernel restart if
  wanted.